# 🧩 Mini-Lab: Baseline, then a 50-Row Gold Set

**Module 9: Is It Any Good? Quality, Latency, Cost** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Run** a majority-class / keyword baseline and record its score
2. **Verify** a 50-row gold set
3. **Compute** accuracy, per-class precision/recall, and a confusion matrix

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Baseline comparison | Judge output against a simple reference, not against "looks good" |
| Gold set | Small verified labelled sample; state the sample-size caveat |
| Precision, recall, confusion matrix | Of the flagged versus of the existing; per-class beats headline; the matrix shows which classes get mixed up |

## Why Start With a Baseline?

Imagine your AI classifier scores **82% accuracy** on support tickets. If **80% of the tickets are "billing"**, a system that *always* guesses "billing" without reading anything also scores 80%. Your model is then only 2 points better than the dumbest possible approach.

A **majority-class baseline** always predicts the most frequent label. If your AI can't beat it, it isn't adding value.

In [1]:
# Simulated data: true labels and AI predictions for 20 support tickets
true_labels = [
    "billing", "billing", "billing", "billing", "billing",
    "billing", "billing", "billing", "billing", "billing",
    "billing", "billing", "billing", "billing",
    "technical", "technical", "technical",
    "account", "account", "account",
]

ai_predictions = [
    "billing", "billing", "billing", "billing", "billing",
    "billing", "billing", "billing", "billing", "billing",
    "billing", "billing", "technical", "account",
    "technical", "technical", "billing",
    "account", "account", "billing",
]

# The distribution is skewed: most tickets are billing
print(f"Total tickets: {len(true_labels)}")
print(f"Label distribution: { {label: true_labels.count(label) for label in set(true_labels)} }")

Total tickets: 20
Label distribution: {'billing': 14, 'account': 3, 'technical': 3}


## Step 1 — Build the Majority-Class Baseline

Find the most common label, then create a "prediction" list that always guesses that label.

> 🧩 **Comparing against a baseline.** Same idea as the video: score the cheap reference first.

In [2]:
from collections import Counter

# Find the majority class
label_counts = Counter(true_labels)
majority_class = label_counts.most_common(1)[0][0]

print(f"Majority class: '{majority_class}' ({label_counts[majority_class]}/{len(true_labels)} tickets)")

# The baseline always predicts the majority class
baseline_predictions = [majority_class] * len(true_labels)

Majority class: 'billing' (14/20 tickets)


## Step 2 — Compare Accuracy: Baseline vs. AI

In [3]:
def accuracy(true, predicted):
    """Fraction of predictions that match the true labels."""
    correct = sum(t == p for t, p in zip(true, predicted))
    return correct / len(true)

baseline_accuracy = accuracy(true_labels, baseline_predictions) * 100
ai_accuracy       = accuracy(true_labels, ai_predictions) * 100
margin            = ai_accuracy - baseline_accuracy

print(f"Baseline accuracy (always guess '{majority_class}'): {baseline_accuracy:.1f}%")
print(f"AI system accuracy:                                  {ai_accuracy:.1f}%")
print(f"Margin over baseline:                                {margin:+.1f} percentage points")

Baseline accuracy (always guess 'billing'): 70.0%
AI system accuracy:                                  80.0%
Margin over baseline:                                +10.0 percentage points


## What to watch

The next cell is a **gold set** — a small, human-verified labeled sample. Watch the size (30 tickets) when you read accuracy: predict whether one headline number will hide a class the model misses.


In [4]:
# 30-row gold set: human-verified labels and the AI's predictions
gold_labels = [
    "billing", "billing", "billing", "billing", "billing",
    "billing", "billing", "billing", "billing", "billing",
    "billing", "billing", "billing", "billing", "billing",
    "technical", "technical", "technical", "technical", "technical",
    "technical", "technical", "technical", "technical",
    "account", "account", "account", "account", "account", "account",
]

ai_predictions = [
    "billing", "billing", "billing", "billing", "billing",
    "billing", "billing", "billing", "billing", "billing",
    "billing", "billing", "technical", "technical", "account",
    "technical", "technical", "technical", "technical", "technical",
    "technical", "billing", "billing", "account",
    "account", "account", "account", "account", "billing", "technical",
]

from collections import Counter
dist = Counter(gold_labels)
print(f"Gold set size: {len(gold_labels)} tickets")
print(f"Label distribution: {dict(dist)}")
print(f"\n⚠️  Sample-size caveat: with only {len(gold_labels)} rows, treat all "
      f"metrics as directional estimates, not exact measurements.")

Gold set size: 30 tickets
Label distribution: {'billing': 15, 'technical': 9, 'account': 6}

⚠️  Sample-size caveat: with only 30 rows, treat all metrics as directional estimates, not exact measurements.


In [5]:
# Reuse accuracy() from the baseline section
ai_acc = accuracy(gold_labels, ai_predictions) * 100

# Majority-class baseline on the gold set
majority_class = Counter(gold_labels).most_common(1)[0][0]
baseline_acc = accuracy(gold_labels, [majority_class] * len(gold_labels)) * 100

margin = ai_acc - baseline_acc

print(f"Majority-class baseline (always '{majority_class}'): {baseline_acc:.1f}%")
print(f"AI accuracy:                                         {ai_acc:.1f}%")
print(f"Margin over baseline:                                {margin:+.1f} pp")

Majority-class baseline (always 'billing'): 50.0%
AI accuracy:                                         73.3%
Margin over baseline:                                +23.3 pp


## Step 3 — Build a Confusion Matrix

Accuracy tells you *how many* errors, but not *what kind*. A **confusion matrix** shows exactly where the AI gets confused: which true class gets mislabeled as which predicted class.

Each row is a **true** label; each column is what the AI **predicted**. Correct predictions sit on the diagonal.

In [6]:
def confusion_matrix(true, predicted, labels):
    """Build a confusion matrix as a dict-of-dicts: matrix[true][predicted] = count."""
    matrix = {t: {p: 0 for p in labels} for t in labels}
    for t, p in zip(true, predicted):
        matrix[t][p] += 1
    return matrix

labels = sorted(set(gold_labels))  # ['account', 'billing', 'technical']
cm = confusion_matrix(gold_labels, ai_predictions, labels)

# Rows = true label, columns = AI prediction. Diagonal = correct.
print("true \\ pred".ljust(12) + "".join(p.rjust(11) for p in labels))
for t in labels:
    print(t.ljust(12) + "".join(str(cm[t][p]).rjust(11) for p in labels))

true \ pred     account    billing  technical
account               4          1          1
billing               1         12          2
technical             1          2          6


## Step 4 — Compute Per-Class Precision and Recall

Watch the next output for per-class **precision** and **recall**, and the confusion matrix behind them. Predict which class the model flags too often versus which it misses.


In [7]:
def precision_recall(cm, labels):
    """Compute per-class precision and recall from a confusion matrix."""
    metrics = {}
    for cls in labels:
        tp = cm[cls][cls]  # true positives: correctly predicted as cls

        # Precision denominator: everything the AI predicted as cls
        predicted_as_cls = sum(cm[t][cls] for t in labels)
        precision = tp / predicted_as_cls if predicted_as_cls > 0 else 0.0

        # Recall denominator: everything that truly was cls
        actually_cls = sum(cm[cls][p] for p in labels)
        recall = tp / actually_cls if actually_cls > 0 else 0.0

        metrics[cls] = {"precision": precision, "recall": recall, "support": actually_cls}
    return metrics

metrics = precision_recall(cm, labels)

# Support = number of true examples in the gold set for that class
for cls in labels:
    m = metrics[cls]
    print(f"{cls:<10} precision={m['precision']:.0%}  recall={m['recall']:.0%}  support={m['support']}")

account    precision=67%  recall=67%  support=6
billing    precision=80%  recall=80%  support=15
technical  precision=67%  recall=67%  support=9


In [8]:
# Gold-set evaluation report: the evidence in one place
print(f"Gold set size:           {len(gold_labels)} tickets")
print(f"Majority-class baseline: {baseline_acc:.1f}%")
print(f"AI accuracy:             {ai_acc:.1f}%")
print(f"Margin over baseline:    {margin:+.1f} pp")
print()
for cls in labels:
    m = metrics[cls]
    print(f"{cls:<10} precision={m['precision']:.0%}  recall={m['recall']:.0%}  support={m['support']}")

# The class the model misses most often
weakest = min(labels, key=lambda cls: metrics[cls]["recall"])

print()
print(f"- Weakest class by recall: '{weakest}' ({metrics[weakest]['recall']:.0%})")
print(f"- ⚠️ Only {len(gold_labels)} rows: these are estimates; expand to 50–100+ rows before a ship decision.")

Gold set size:           30 tickets
Majority-class baseline: 50.0%
AI accuracy:             73.3%
Margin over baseline:    +23.3 pp

account    precision=67%  recall=67%  support=6
billing    precision=80%  recall=80%  support=15
technical  precision=67%  recall=67%  support=9

- Weakest class by recall: 'account' (67%)
- ⚠️ Only 30 rows: these are estimates; expand to 50–100+ rows before a ship decision.


## 🎯 Summary

### Key Takeaways

1. **Gold set** — a small, human-verified labeled dataset that serves as ground truth; always state its size alongside any metric you report
2. **Confusion matrix** — shows exactly which classes get confused with which, going beyond a single accuracy number
3. **Precision** — of everything the AI predicted as class X, how many were actually X (measures false-positive rate)
4. **Recall** — of everything that truly was class X, how many did the AI catch (measures false-negative rate)
5. **Baseline comparison** — accuracy only means something when compared to the majority-class baseline; always report the margin